# Xenium: TIL density against ECM and cell types in 55 µm tiles (Fig 6b, 6g)

Produces, for both Xenium gene panels (hBrain_v1.2 with the protein panel, and hAtlas_v1.1, the 5K panel), six sections each:

- **Fig 6g** (`celltype_level4_curated_first_type_spearman_tiles`): per-section Spearman correlation between the TIL count of a tile and its count of each level-4 cell type.
- **Fig 6b** (`ecm_spearman_tiles`): per-section Spearman correlation between the TIL count of a tile and its ECM category score (decoupler ULM on the tile pseudobulk, matrisome categories of Supp Table 4a).

Tiles with at least 5 cells and at least one TIL, sections with at least 10 such tiles, Benjamini-Hochberg within each section. Segmentation: 10x multimodal with 5 µm expansion (`10x_mm_5um`); counts corrected with SPLIT (`split_fully_purified`), RCTD labels of the raw counts. The Methods text says Proseg segmentation; the published panels used `10x_mm_5um`.

**Origin**: `analysis/xenium/niches.ipynb` (j-bac), cells 0, 2, 4, 6, 28, 31 and 33, run for both panels in one notebook.

**Compared with the published panels**: the CSVs written with the published figures (EFS `figures/xenium_fig_6.zip`) are reproduced exactly for both panels (same rows and n_tiles; rho, p and p_adj identical; checked at the end).

In [ ]:
import time
import zipfile
from pathlib import Path

import decoupler as dc
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

from gbmcollaboration.paper import config, output
from gbmcollaboration.utils import xenium_io_utils as xio
from gbmcollaboration.viz.dotplot import dotplot

## Parameters


In [ ]:
OUT = output.setup("fig6bg_xenium_niches")

# path to ecm metadata
path_ecm_csv = config.EFS / 'supplementary_tables/4a_ECM_genes.csv'

# path to raw and processed xenium
base_dir = config.XENIUM_BASE

# folder structure
xenium_levels = ['segmentation','condition','panel','donor','sample']

# data parameters
correction_methods = ['raw','split_fully_purified'] # correction of count matrices after segmentation
segmentation = '10x_mm_5um' # or '10x_mm_5um' or 'proseg_expected' or 'all'
condition = 'gbm'
panels = ['hBrain_v1.2', 'hAtlas_v1.1']  # both panels of the paper

# cell type annotation parameters
normalisation = 'lognorm'
reference = 'allcenters_10k_ct'  # annotated with all centers snRNA ref, subsampled to 10k max per cell type
method = 'rctd_class_aware'
level = None # celltype_curated_level1,2,3,4 available
level_keys = [f'celltype_level{i}_curated_first_type' for i in range(1,5)]

# SPLIT parameters
split_level='celltype_level2_curated'

# fixed params
BATCH_KEY = 'dataset_id'
SPATIAL_KEY = 'spatial'

read_kwargs = dict(
    # set anndata = False to load in spatialdata format instead of a single anndata
    # (allows to load images, transcripts, etc)
    anndata = True,

    # extra parameters if anndata = False
    cells_as_circles = False,
    cells_boundaries = False,
    cells_boundaries_layers = False, # for proseg only
    nucleus_boundaries = False,
    cells_labels = False,
    nucleus_labels = False,
    transcripts = False,
    morphology_mip = False,
    morphology_focus = False,
    aligned_images = False,
    gex_only = True,
    pool_mode="thread",
    max_workers=6
)

# published panels (comparison at the end)
published_zip = config.EFS / 'figures/xenium_fig_6.zip'

In [ ]:
def pseudobulk_lognorm(adata, by):
    pb = sc.get.aggregate(adata, by=by, func='sum',layer='counts' )
    pb.X = pb.layers['sum']
    sc.pp.normalize_total(pb)
    sc.pp.log1p(pb)
    return pb


def assign_grid_tiles(adata, sample_key='sample', spatial_key='spatial', tile_size=55.0):
    """
    Assigns each cell to a non-overlapping spatial grid tile.
    Adds 'tile_id' to adata.obs.
    """
    x = adata.obsm[spatial_key][:, 0]
    y = adata.obsm[spatial_key][:, 1]

    # Calculate discrete grid indices
    tile_x = np.floor(x / tile_size).astype(int).astype(str)
    tile_y = np.floor(y / tile_size).astype(int).astype(str)

    # Create unique tile IDs. We prepend the sample ID
    samples = adata.obs[sample_key].astype(str)
    tile_ids = samples + "_tile_" + tile_x + "_" + tile_y

    adata.obs['tile_id'] = pd.Categorical(tile_ids)


def spearman_tils(pb, categories, sample_key, til_count_key='TIL_count',
                  min_tiles=10, filter_zero_til=True):
    """
    Computes Spearman correlation per sample and applies FDR correction per sample.
    """
    # 1. Prepare data
    pb_df = pb.obs.copy() # Work directly on obs DataFrame
    if filter_zero_til:
        pb_df = pb_df[pb_df[til_count_key] > 0]

    # 2. Correlate per sample
    results = []
    for sample_id, group in pb_df.groupby(sample_key, observed=True):
        if len(group) < min_tiles:
            continue

        for cat in categories:
            rho, p_val = spearmanr(group[til_count_key], group[cat], nan_policy='omit')
            results.append({
                sample_key: sample_id,
                'category': cat,
                'n_tiles': len(group),
                'rho': rho,
                'p': p_val
            })

    res_df = pd.DataFrame(results)


    # 3. Correct p-values per sample
    def fdr_correct(group):
        # Drop NaNs before correction
        valid = group['p'].notna()
        p_adj = np.full(len(group), np.nan)
        if valid.any():
            _, corrected, _, _ = multipletests(group.loc[valid, 'p'], method='fdr_bh')
            p_adj[valid] = corrected
        group['p_adj'] = p_adj
        return group

    res_df = res_df.groupby(sample_key, group_keys=False).apply(fdr_correct)

    return res_df


def load_panel(panel):
    xenium_paths, xenium_annot_paths = xio.discover_xenium_paths(
        base_dir=base_dir,
        normalisation=normalisation,
        reference=reference,
        method=method,
        level=None,
        load_correction=True,
        split_level='celltype_level2_curated',
        segmentations_filter=[segmentation],
        conditions_filter=[condition] if condition != 'all' else None,
        panels_filter=[panel] if panel != 'all' else None,
    )

    # Read samples
    xds = xio.read_count_correction_samples(xenium_paths,['split_fully_purified'])
    xds['raw'] = xio.read_xenium_samples(xenium_paths['raw'],**read_kwargs)

    # add cell type annotation from raw to all correction methods
    xio.read_annotations(xds, correction_methods, xenium_annot_paths, level, max_workers=read_kwargs['max_workers'])

    for corr, xds_corr in xds.items():
        for k, xd in xds_corr.items():
            print(corr,k)
            xd.layers['counts'] = xd.X.copy()

            if corr != 'raw':
                xd.obsm['spatial'] = xds['raw'][k][xd.obs_names].obsm['spatial']

    # concat samples into a single anndata
    adata_dict = {}
    for corr in correction_methods[1:]:
        ad_ = sc.concat({k: v for k, v in xds[corr].items() },label='dataset_id', join='outer')
        ad_.obs[xenium_levels] = pd.DataFrame(ad_.obs['dataset_id'].tolist(),index=ad_.obs.index,columns=xenium_levels)
        ad_.obs['correction_method'] = corr
        adata_dict[corr] = {'rna':ad_}

    # info about the loaded cohort
    df_cohort_info = pd.DataFrame(
        [k+(corr_method,)
        for corr_method in correction_methods
        for k in xds[corr_method].keys() ],
        columns=xenium_levels+['correction_method']
    )
    print(f'Loaded {len(df_cohort_info)} samples:')
    display(df_cohort_info)
    return adata_dict


def prepare_panel(adata_dict):
    # get adatas per modality and remove cells with NaN annotations
    ad_rna =  adata_dict[corr]['rna']
    has_annotation_idx = ad_rna.obs[level_keys].notna().all(1)

    ad_rna = ad_rna[has_annotation_idx].copy()

    # prep .obs
    for k in [sample_key,cluster_key]:
        ad_rna.obs[k] = ad_rna.obs[k].astype('category')

    ad_rna.obs['TIL_status_bool'] = ad_rna.obs[cluster_key].isin(til_cell_types)
    ad_rna.obs['TIL_status'] = np.where(ad_rna.obs['TIL_status_bool'],'TIL','non-TIL')

    # assign grid tiles
    assign_grid_tiles(ad_rna, sample_key=sample_key, spatial_key='spatial', tile_size=55.0)
    return ad_rna


def tile_panel(ad_rna):
    # Calculate tile-level metadata from the single-cell adata
    tile_meta = ad_rna.obs.groupby('tile_id', observed=True).agg(
        total_cells=('tile_id', 'size'),
        TIL_count=(til_key, 'sum')
    )
    cluster_counts = (
        ad_rna.obs.groupby('tile_id', observed=True)[cell_type_key]
        .value_counts()
        .unstack(fill_value=0)
    )
    tile_meta = tile_meta.join(cluster_counts)

    # pseudobulk by tile
    pb = pseudobulk_lognorm(ad_rna, by=[sample_key,'tile_id'])

    # Filter out tiles with fewer than n cells
    valid_tiles = tile_meta[tile_meta['total_cells'] >= min_n_cells_per_tile].index
    tile_meta = tile_meta.loc[valid_tiles]
    pb = pb[pb.obs['tile_id'].isin(valid_tiles)].copy()

    # Add the cell counts and TIL counts back into the pseudobulked AnnData.obs
    pb.obs = pb.obs.join(tile_meta,on='tile_id')

    # Calculate ECM ULM scores
    dc.run_ulm(pb,net=df_ecm,min_n=1,use_raw=False,weight=None)
    df_ulm = pb.obsm['ulm_estimate']
    pb.obs = pb.obs.assign(**df_ulm)

    # Categories to test
    cell_type_cats = ad_rna.obs[cell_type_key].unique().tolist()
    ecm_cats = df_ulm.columns
    donors = ad_rna.obs.set_index(sample_key)['donor'].drop_duplicates()
    return pb, cell_type_cats, ecm_cats, donors


def compare(new, zip_member):
    with zipfile.ZipFile(published_zip) as z, z.open(zip_member) as f:
        old = pd.read_csv(f, index_col=0)
    key = [sample_key, 'category']
    m = old.merge(new, on=key, how='outer', suffixes=('_pub', '_new'), indicator=True)
    both = m[m['_merge'] == 'both']
    out = {'rows published': len(old), 'rows here': len(new),
           'only published': int((m['_merge'] == 'left_only').sum()),
           'only here': int((m['_merge'] == 'right_only').sum()),
           'n_tiles equal': bool((both['n_tiles_pub'] == both['n_tiles_new']).all())}
    for c in ['rho', 'p', 'p_adj']:
        a, b = both[f'{c}_pub'].to_numpy(float), both[f'{c}_new'].to_numpy(float)
        out[f'max |Δ{c}|'] = float(np.nanmax(np.abs(a - b))) if len(a) else np.nan
        out[f'{c} equal'] = bool(np.allclose(a, b, rtol=1e-9, atol=0, equal_nan=True))
    return out

### prepare data for analysis

In [ ]:
# analysis parameters
corr = 'split_fully_purified'
til_cell_types = ['T_cell', 'NK']
sample_key = 'sample'
cluster_key = level_keys[2]

# read ecm metadata
df_ecm = pd.read_csv(path_ecm_csv)

## Spearman pseudobulk tiles

In [ ]:
cell_type_key = level_keys[-1]
til_key = 'TIL_status_bool'
min_n_cells_per_tile = 5
min_n_tiles_per_sample = 10

# per panel; keep only the tile pseudobulk
tiles = {}
for panel in panels:
    t0 = time.time()
    adata_dict = load_panel(panel)
    ad_rna = prepare_panel(adata_dict)
    del adata_dict
    tiles[panel] = tile_panel(ad_rna)
    pb, cell_type_cats, ecm_cats, donors = tiles[panel]
    print(f"{panel}: {ad_rna.obs[sample_key].nunique()} sections, {ad_rna.n_obs:,} annotated cells, "
          f"{pb.n_obs:,} tiles with >= {min_n_cells_per_tile} cells ({int((pb.obs['TIL_count'] > 0).sum()):,} with TIL > 0), "
          f"{len(cell_type_cats)} level-4 cell types, ECM categories {list(ecm_cats)} ({time.time() - t0:.0f} s)")
    del ad_rna

### cell type composition (Fig 6g)

In [ ]:
fig6g = {}
for panel in panels:
    pb, cell_type_cats, ecm_cats, donors = tiles[panel]

    # correlate filtering out TIL count = 0
    res_df = spearman_tils(pb, cell_type_cats, sample_key, til_count_key='TIL_count', min_tiles=min_n_tiles_per_sample, filter_zero_til=True)
    res_df = res_df.join(donors,on=sample_key)

    # plot
    x_order = res_df.groupby('category')['rho'].mean().sort_values(ascending=False).index

    ax = dotplot(res_df,
        x_col='category',
        y_col='donor',
        diff_col='rho',
        pval_col='p_adj',
        x_order=x_order,
        y_order=np.unique(res_df['donor']),
        figsize=(18, 5),
        # yticklabels=False,
        cbar_title='Spearman ρ',
        bbox_to_anchor_dot = (1.08, .8),
        min_dot_size=50,
        max_dot_size=200,
        center=0,
        save_path=None
    )
    ax.set_title(f'Fig 6g, {panel}')
    (OUT / panel).mkdir(exist_ok=True)
    output.save(OUT / panel, f'{cell_type_key}_spearman_tiles', fig=ax.figure, data=res_df,
                panel=f"Fig 6g {panel.split('_')[0]}")
    plt.show()
    fig6g[panel] = res_df

### ECM (Fig 6b)

In [ ]:
fig6b = {}
for panel in panels:
    pb, cell_type_cats, ecm_cats, donors = tiles[panel]

    # correlate filtering out TIL count = 0
    res_df = spearman_tils(pb, ecm_cats, sample_key, til_count_key='TIL_count', min_tiles=min_n_tiles_per_sample, filter_zero_til=True)
    res_df = res_df.join(donors,on=sample_key)

    # plot
    x_order = res_df.groupby('category')['rho'].mean().sort_values(ascending=False).index

    ax = dotplot(res_df,
        x_col='category',
        y_col='donor',
        diff_col='rho',
        pval_col='p_adj',
        x_order=x_order,
        y_order=np.unique(res_df['donor']),
        figsize=(10, 5),
        # yticklabels=False,
        cbar_title='Spearman ρ',
        bbox_to_anchor_dot = (1.15, .8),
        # min_dot_size=50,
        # max_dot_size=400,
        save_path=None
    )
    ax.set_title(f'Fig 6b, {panel}')
    (OUT / panel).mkdir(exist_ok=True)
    output.save(OUT / panel, 'ecm_spearman_tiles', fig=ax.figure, data=res_df,
                panel=f"Fig 6b {panel.split('_')[0]}")
    plt.show()
    fig6b[panel] = res_df
    print(panel)
    print(res_df.pivot(index='donor', columns='category', values='rho')[x_order].round(3).to_string())

## Check against the published panels

The CSVs written with the published figures (EFS `figures/xenium_fig_6.zip`, `xenium/<panel>/10x_mm_5um/`) are compared row by row on (section, category): n_tiles exactly, rho, p and p_adj to 1e-9 relative.

In [ ]:
rows = {}
for panel in panels:
    base = f'xenium/{panel}/{segmentation}/'
    rows[(panel, 'Fig 6b')] = compare(fig6b[panel], base + 'ecm_spearman_tiles.csv')
    rows[(panel, 'Fig 6g')] = compare(fig6g[panel], base + f'{cell_type_key}_spearman_tiles.csv')
check = pd.DataFrame(rows).T
output.save_table(OUT, 'check_vs_published', check)
check

In [ ]:
print(output.versions(["numpy", "pandas", "scipy", "statsmodels", "anndata", "scanpy", "decoupler",
                       "spatialdata", "matplotlib"]).to_string(index=False))